# 🌍 GIEC Chat - AR6 (RAG avec citations)

**Projet IA / Climat - Rapport GIEC AR6 WG1**

Notebook unique regroupant tout le pipeline :
1. **Ingestion** PDF -> chunks
2. **Vectorisation** chunks -> FAISS (bge-m3)
3. **Requetage** RAG + App Streamlit

> Donnees : https://www.ipcc.ch/report/ar6/wg1/ -> Placer `IPCC_AR6_WGI_Chapter01.pdf` dans `data/raw/`


## 0. Mettre Google drive comme répertoire de travail par défaut

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/Climate Change - GIEC chat"

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/Climate Change - GIEC chat


## 1. Installation
A exécuter une fois

In [ ]:
# Installation des dépendances (sans langchain) - À EXÉCUTER EN PREMIER
%pip install -q pypdf sentence-transformers faiss-cpu streamlit requests
print("Dépendances installées (pypdf + sentence-transformers + faiss)")
try:
    import pypdf, sentence_transformers, faiss
    print("✅ Vérif imports OK - tu peux passer à la cellule suivante")
except Exception as e:
    print(f"⚠️ Relance le runtime si besoin: {e}")


In [2]:
import json
import re
import requests
from pathlib import Path
try:
    from pypdf import PdfReader
except ModuleNotFoundError:
    import subprocess, sys
    print("[INFO] pypdf manquant -> pip install ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pypdf"])
    from pypdf import PdfReader
    print("✅ pypdf installé")

# --- Config multi-chapitres (1-12 + SPM) ---
BASE_URL = "https://www.ipcc.ch/report/ar6/wg1/downloads/report/"
CHAPTERS = {
    1: "IPCC_AR6_WGI_Chapter01.pdf",
    2: "IPCC_AR6_WGI_Chapter02.pdf",
    3: "IPCC_AR6_WGI_Chapter03.pdf",
    4: "IPCC_AR6_WGI_Chapter04.pdf",
    5: "IPCC_AR6_WGI_Chapter05.pdf",
    6: "IPCC_AR6_WGI_Chapter06.pdf",
    7: "IPCC_AR6_WGI_Chapter07.pdf",
    8: "IPCC_AR6_WGI_Chapter08.pdf",
    9: "IPCC_AR6_WGI_Chapter09.pdf",
    10: "IPCC_AR6_WGI_Chapter10.pdf",
    11: "IPCC_AR6_WGI_Chapter11.pdf",
    12: "IPCC_AR6_WGI_Chapter12.pdf",
    "SPM": "IPCC_AR6_WGI_SPM.pdf",
}
CHAPTER_TITLES = {
    1: "Ch.1 - Smoking gun", 2: "Ch.2 - Changing state", 3: "Ch.3 - Human influence",
    4: "Ch.4 - Future climate", 5: "Ch.5 - Carbon cycle", 6: "Ch.6 - Extremes",
    7: "Ch.7 - Clouds & aerosols", 8: "Ch.8 - Water cycle", 9: "Ch.9 - Ocean & cryosphere",
    10: "Ch.10 - Regional", 11: "Ch.11 - Extremes regional", 12: "Ch.12 - Climate information for risk",
    "SPM": "Summary for Policymakers"
}

def extract_with_metadata(pdf_path: Path, chapter_id):
    """Extrait chaque page avec pypdf, enrichit metadata avec chapter."""
    reader = PdfReader(str(pdf_path))
    chunks = []
    title = CHAPTER_TITLES.get(chapter_id, f"Ch.{chapter_id}")
    source_label = f"AR6 WG1 Ch.{chapter_id}" if isinstance(chapter_id, int) else f"AR6 WG1 {chapter_id}"
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r'\n{2,}', '\n', text).strip()
        if len(text) < 300:
            continue
        chunks.append({
            "text": text,
            "metadata": {
                "source": source_label,
                "chapter": chapter_id,
                "chapter_title": title,
                "page": i + 1,
            }
        })
    return chunks

def decouper(texte, taille=800, chevauchement=100):
    """Découpe sans langchain : ~800 chars, overlap 100, coupe en fin de phrase."""
    texte = " ".join(texte.split())
    passages, debut = [], 0
    while debut < len(texte):
        fin = min(debut + taille, len(texte))
        if fin < len(texte):
            coupe = texte.rfind(". ", debut + taille // 2, fin)
            if coupe != -1:
                fin = coupe + 1
        passages.append(texte[debut:fin].strip())
        if fin >= len(texte):
            break
        debut = max(fin - chevauchement, debut + 1)
    return [p for p in passages if p]

def split_chunks(chunks, chunk_size=800, chunk_overlap=100):
    final = []
    for c in chunks:
        for chunk in decouper(c["text"], taille=chunk_size, chevauchement=chunk_overlap):
            final.append({"text": chunk, "metadata": c["metadata"]})
    return final

# --- Boucle full 1-12 + SPM ---
ALL_CHUNKS = []
for ch_id, fname in CHAPTERS.items():
    pdf_path = Path(f"data/raw/{fname}")
    # naming conservé : data/processed/chunks_chapterX.json
    out_name = f"chunks_chapter{ch_id}.json" if isinstance(ch_id, int) else f"chunks_{str(ch_id).lower()}.json"
    out_path = Path(f"data/processed/{out_name}")
    print(f"\n--- Chapitre {ch_id} : {fname} ---")
    if not pdf_path.exists():
        url = BASE_URL + fname
        print(f"[INFO] PDF non trouvé: {pdf_path} -> téléchargement {url} ...")
        pdf_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            pdf_path.write_bytes(r.content)
            print(f"✅ Téléchargé : {pdf_path} ({len(r.content)/1e6:.1f} MB)")
        except Exception as e:
            print(f"[WARN] Échec téléchargement Ch.{ch_id} : {e} -> skip")
            continue
    else:
        print(f"✅ PDF présent : {pdf_path}")
    print(f"Extraction de {pdf_path} ...")
    raw = extract_with_metadata(pdf_path, ch_id)
    print(f"  {len(raw)} pages extraites")
    final = split_chunks(raw)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(final, f, ensure_ascii=False, indent=2)
    print(f"  {len(final)} chunks -> {out_path}")
    ALL_CHUNKS.extend(final)

# Agrégat unifié pour l'index unique (non committé)
ALL_PATH = Path("data/processed/chunks_all.json")
with open(ALL_PATH, "w", encoding="utf-8") as f:
    json.dump(ALL_CHUNKS, f, ensure_ascii=False, indent=2)
print(f"\n✅ Total {len(ALL_CHUNKS)} chunks (tous chapitres) -> {ALL_PATH}")
# Stats par chapitre
from collections import Counter
cnt = Counter(c["metadata"]["chapter"] for c in ALL_CHUNKS)
for k in sorted(cnt, key=lambda x: (isinstance(x,str), x)):
    print(f"  Ch.{k}: {cnt[k]} chunks")


ModuleNotFoundError: No module named 'pypdf'

## 2. Ingestion - PDF -> chunks
Découpe le Chapitre 1 en chunks de 800 tokens avec overlap 100. Conserve la page source pour citation.


In [ ]:
import json
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Source unifiée (tous chapitres) — per-chapter conservés en data/processed/chunks_chapter*.json
CHUNKS_PATH = Path("data/processed/chunks_all.json")
# Fallback si chunks_all pas encore généré (ex: run Ch.1 seul)
if not CHUNKS_PATH.exists():
    fallback = Path("data/processed/chunks_chapter1.json")
    print(f"[INFO] {CHUNKS_PATH} absent -> fallback {fallback}")
    CHUNKS_PATH = fallback
VECTORSTORE_PATH = Path("vectorstore/giec_wg1_all")
EMBEDDING_MODEL = "BAAI/bge-m3"

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(f"[ERREUR] {CHUNKS_PATH} introuvable. Lance d'abord la cellule 4 (ingestion full).")

with open(CHUNKS_PATH, encoding="utf-8") as f:
    data_json = json.load(f)

texts = [d["text"] for d in data_json]
metadatas = [d["metadata"] for d in data_json]

print(f"Vectorisation de {len(texts)} chunks (tous chapitres) avec {EMBEDDING_MODEL} ...")
model = SentenceTransformer(EMBEDDING_MODEL)
# batch_size=64 conservé
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=64)
embeddings = np.array(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
VECTORSTORE_PATH.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(VECTORSTORE_PATH / "index.faiss"))
with open(VECTORSTORE_PATH / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(data_json, f, ensure_ascii=False, indent=2)
# Stats par chapitre pour vérif
from collections import Counter
cnt = Counter(m["chapter"] for m in metadatas)
print(f"Index FAISS unifié -> {VECTORSTORE_PATH} ({index.ntotal} vecteurs, {embeddings.shape[1]} dims)")
for k in sorted(cnt, key=lambda x: (isinstance(x,str), x)):
    print(f"  Ch.{k}: {cnt[k]} chunks")

# Test cross-chapitres
for query in ["Quelle est la part du réchauffement d'origine humaine ?", "Probabilité de dépasser 1.5°C avant 2040 ?"]:
    q_emb = model.encode([query], normalize_embeddings=True)
    scores, ids = index.search(np.array(q_emb, dtype="float32"), k=3)
    print(f"\nTest: {query}")
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), 1):
        print(f"  {rank}. [score={score:.3f}] Ch.{metadatas[idx]['chapter']} p.{metadatas[idx]['page']} {metadatas[idx]['source']}")
        print(f"     {texts[idx][:180]}...")


Extraction de data/raw/IPCC_AR6_WGI_Chapter01.pdf ...
  140 pages extraites
  1341 chunks créés -> data/processed/chunks_chapter1.json


In [ ]:
import os

print(os.path.exists("data/processed/chunks_chapter1.json"))
print(os.path.abspath("data/processed/chunks_chapter1.json"))

True
/content/drive/MyDrive/Colab Notebooks/Climate Change - GIEC chat/data/processed/chunks_chapter1.json


import json
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

VECTORSTORE_PATH = Path("vectorstore/giec_wg1_all")
# Fallback ancien single-chapitre
if not (VECTORSTORE_PATH / "index.faiss").exists():
    VECTORSTORE_PATH = Path("vectorstore/giec_wg1_ch1")
    print(f"[INFO] giec_wg1_all absent -> fallback {VECTORSTORE_PATH}")
EMBEDDING_MODEL = "BAAI/bge-m3"

if not (VECTORSTORE_PATH / "index.faiss").exists():
    print("Index non trouvé. Lance les cellules 4 et 6 (full ingestion) d'abord.")
else:
    index = faiss.read_index(str(VECTORSTORE_PATH / "index.faiss"))
    with open(VECTORSTORE_PATH / "chunks.json", encoding="utf-8") as f:
        data = json.load(f)
    texts = [d["text"] for d in data]
    metadatas = [d["metadata"] for d in data]
    model = SentenceTransformer(EMBEDDING_MODEL)
    for query in ["Quelle est la part du réchauffement d'origine humaine ?", "Hausse du niveau de la mer d'ici 2100 ?"]:
        q_emb = model.encode([query], normalize_embeddings=True)
        scores, ids = index.search(np.array(q_emb, dtype="float32"), k=5)
        print(f"\n--- Query: {query} ---")
        for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), 1):
            print(f"{rank}. [score={score:.3f}] Ch.{metadatas[idx]['chapter']} p.{metadatas[idx]['page']} {metadatas[idx]['source']}")
            print(texts[idx][:500])
            print()


In [ ]:
import json
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

CHUNKS_PATH = Path("data/processed/chunks_chapter1.json")
VECTORSTORE_PATH = Path("vectorstore/giec_wg1_ch1")
EMBEDDING_MODEL = "BAAI/bge-m3" # bge-m3 = modèle multilingue (FR/EN, 100+ langues), 560M paramètres, vecteur de 1024 dimensions (embeddings_chunks.shape[1]).

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(f"[ERREUR] {CHUNKS_PATH} introuvable. Lance d'abord la cellule 4 (ingestion).")

with open(CHUNKS_PATH, encoding="utf-8") as f:
    data = json.load(f)

texts = [d["text"] for d in data]
metadatas = [d["metadata"] for d in data]

print(f"Vectorisation de {len(texts)} chunks avec {EMBEDDING_MODEL} ...")
model = SentenceTransformer(EMBEDDING_MODEL)
embeddings_chunks = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=64) # on normalise en norme L2 = la similarité cosinus devient un simple produit scalaire. Indispensable pour FAISS IndexFlatIP.
embeddings_chunks = np.array(embeddings_chunks, dtype="float32")

# ~1500 chunks du Ch.1 GIEC -> matrice 1500 x 1024 (~6 MB).

# FAISS natif (cosine = IndexFlatIP car embeddings normalisés) (Facebook AI Similarity Search) est une base de vecteurs
index = faiss.IndexFlatIP(embeddings_chunks.shape[1])
index.add(embeddings_chunks)
VECTORSTORE_PATH.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(VECTORSTORE_PATH / "index.faiss"))
# Sauvegarde textes + métadonnées pour la recherche
with open(VECTORSTORE_PATH / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
print(f"Index FAISS sauvegardé -> {VECTORSTORE_PATH} ({index.ntotal} vecteurs)")

# Test rapide
question = "Quelle est la part du réchauffement d'origine humaine observé ?"
question_embeddings = model.encode([question], normalize_embeddings=True)
scores, ids = index.search(np.array(question_embeddings, dtype="float32"), k=3)
print(f"\nTest question: {question}")
for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), 1):
    print(f"\n--- Résultat {rank} [score={score:.3f}] {metadatas[idx]} ---")
    print(texts[idx][:500])


Vectorisation de 1341 chunks avec BAAI/bge-m3 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (10 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Index FAISS sauvegardé -> vectorstore/giec_wg1_ch1 (1341 vecteurs)

Test question: Quelle est la part du réchauffement d'origine humaine observé ?

--- Résultat 1 [score=0.697] {'source': 'AR6 WG1 Chapitre 1', 'page': 38} ---
t of the warming observed over the last 50 years is attributable to human activities’ (IPCC, 2001b). The AR4 further strengthened previous statements, concluding that ‘most of the observed increase in global average temperatures since the mid-20th century is very likely due to the observed increase in anthropogenic greenhouse gas concentrations’ (IPCC, 2007b). The AR5 assessed that a human contribution had been detected in: changes in warming of the atmosphere and ocean; changes in the global wa

--- Résultat 2 [score=0.684] {'source': 'AR6 WG1 Chapitre 1', 'page': 140} ---
e anthropogenic increase in greenhouse gas concentrations and other anthropogenic forcings together. The best estimate of the human-induced contribution to warming is similar to the observed wa

# Code complet app.py - Streamlit unifié 1-12+SPM (sans langchain)
import json
import faiss
import numpy as np
import streamlit as st
from pathlib import Path
from sentence_transformers import SentenceTransformer

VECTORSTORE_PATH = Path("vectorstore/giec_wg1_all")
if not (VECTORSTORE_PATH / "index.faiss").exists():
    VECTORSTORE_PATH = Path("vectorstore/giec_wg1_ch1")
EMBEDDING_MODEL = "BAAI/bge-m3"

SYSTEM_PROMPT = """Tu es l'assistant GIEC. Réponds UNIQUEMENT avec le contexte fourni.
Cite obligatoirement [Source: WG1 Ch.X/SPM, p.Y]. Si la réponse n'est pas dans le contexte, dis "Non trouvé dans le rapport". N'invente rien."""

st.set_page_config(page_title="GIEC Chat - AR6 WG1 (1-12+SPM)", layout="wide")
st.title("🌍 GIEC Chat - AR6 WG1 (Ch.1-12 + SPM)")
st.caption("RAG unifié avec citations vérifiables - Données: IPCC AR6 WG1")

@st.cache_resource
def load_db():
    if not (VECTORSTORE_PATH / "index.faiss").exists():
        return None, None, None, None
    index = faiss.read_index(str(VECTORSTORE_PATH / "index.faiss"))
    with open(VECTORSTORE_PATH / "chunks.json", encoding="utf-8") as f:
        data = json.load(f)
    texts = [d["text"] for d in data]
    metadatas = [d["metadata"] for d in data]
    model = SentenceTransformer(EMBEDDING_MODEL)
    return index, texts, metadatas, model

index, texts, metadatas, model = load_db()

if index is None:
    st.warning("Index unifié non trouvé. Lance les cellules 4 et 6 (full 1-12+SPM) après téléchargement des PDFs dans `data/raw/`.")
    st.info("PDFs auto-téléchargés depuis: https://www.ipcc.ch/report/ar6/wg1/downloads/report/")
    st.stop()

# Filtre chapitre
chapters = sorted(set(m["chapter"] for m in metadatas), key=lambda x: (isinstance(x,str), x))
sel = st.selectbox("Filtrer par chapitre", ["Tous"] + [f"Ch.{c}" for c in chapters])
query = st.text_input("Pose ta question au rapport du GIEC :", placeholder="Ex: Quelle est la probabilité de dépasser 1.5°C avant 2040 ?")

if query:
    q_emb = model.encode([query], normalize_embeddings=True)
    # k=5 pour couvrir plusieurs chapitres
    scores, ids = index.search(np.array(q_emb, dtype="float32"), k=10)
    # Filtrage si sélection
    docs = []
    for rank, idx in enumerate(ids[0]):
        meta = metadatas[idx]
        if sel != "Tous" and f"Ch.{meta['chapter']}" != sel:
            continue
        docs.append((texts[idx], meta, float(scores[0][rank])))
        if len(docs) >= 5:
            break
    col1, col2 = st.columns([2, 1])
    with col1:
        st.subheader("Réponse (à brancher à un LLM)")
        st.info(f"{SYSTEM_PROMPT}\n\nContexte récupéré pour: '{query}' (filtre: {sel})")
        for i, (text, meta, score) in enumerate(docs, 1):
            st.markdown(f"**Source {i} - Ch.{meta['chapter']} p.{meta['page']} [score={score:.3f}]**")
            st.write(text)
            st.divider()
    with col2:
        st.subheader("Sources vérifiables")
        for _, meta, _ in docs:
            st.markdown(f"- **{meta['source']}**, p.{meta['page']} ({meta['chapter_title']})")
        st.caption("Index unifié giec_wg1_all - Branche un LLM (Mistral) pour génération avec citations.")


In [ ]:
import json
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

VECTORSTORE_PATH = Path("vectorstore/giec_wg1_ch1")
EMBEDDING_MODEL = "BAAI/bge-m3"

if not (VECTORSTORE_PATH / "index.faiss").exists():
    print("Index non trouvé. Lance les cellules 4 et 6 d'abord.")
else:
    index = faiss.read_index(str(VECTORSTORE_PATH / "index.faiss"))
    with open(VECTORSTORE_PATH / "chunks.json", encoding="utf-8") as f:
        data = json.load(f)
    texts = [d["text"] for d in data]
    metadatas = [d["metadata"] for d in data]
    model = SentenceTransformer(EMBEDDING_MODEL)
    question = "Quelle est la part du réchauffement d'origine humaine ?"
    question_embeddings = model.encode([question], normalize_embeddings=True)
    scores, ids = index.search(np.array(question_embeddings, dtype="float32"), k=3)
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), 1):
        print(f"--- Resultat {rank} [score={score:.3f}] {metadatas[idx]} ---")
        print(texts[idx][:600])
        print()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

--- Resultat 1 [score=0.677] {'source': 'AR6 WG1 Chapitre 1', 'page': 140} ---
e anthropogenic increase in greenhouse gas concentrations and other anthropogenic forcings together. The best estimate of the human-induced contribution to warming is similar to the observed warming over this period. It is unequivocal that human influence has warmed the atmosphere, ocean and land. The likely range of total human-caused global surface temperature increase from 1850–1900 to 2010–201911 is 0.8°C to 1.3°C, with a best estimate of 1.07°C. It is likely that well-mixed GHGs contributed a warming of 1.0°C to 2.0°C, other human drivers (principally aerosols) contributed a cooling of 0.

--- Resultat 2 [score=0.670] {'source': 'AR6 WG1 Chapitre 1', 'page': 38} ---
t of the warming observed over the last 50 years is attributable to human activities’ (IPCC, 2001b). The AR4 further strengthened previous statements, concluding that ‘most of the observed increase in global average temperatures since the mi

## 5. App Streamlit (optionnel - pour deploiement local)
Code complet pour `streamlit run app.py`. Copie integree ci-dessous pour reference.


In [ ]:
# Code complet app.py - reference / deploiement Streamlit (sans langchain)
import json
import faiss
import numpy as np
import streamlit as st
from pathlib import Path
from sentence_transformers import SentenceTransformer

VECTORSTORE_PATH = Path("vectorstore/giec_wg1_ch1")
EMBEDDING_MODEL = "BAAI/bge-m3"

SYSTEM_PROMPT = """Tu es l'assistant GIEC. Réponds UNIQUEMENT avec le contexte fourni.
Cite obligatoirement [Source: WG1 Ch1, p.X]. Si la réponse n'est pas dans le contexte, dis "Non trouvé dans le rapport". N'invente rien."""

st.set_page_config(page_title="GIEC Chat - AR6 WG1", layout="wide")
st.title("🌍 GIEC Chat - AR6 WG1 Chapitre 1")
st.caption("RAG avec citations vérifiables - Données: IPCC AR6 WG1 (sans langchain)")

@st.cache_resource
def load_db():
    if not (VECTORSTORE_PATH / "index.faiss").exists():
        return None, None, None, None
    index = faiss.read_index(str(VECTORSTORE_PATH / "index.faiss"))
    with open(VECTORSTORE_PATH / "chunks.json", encoding="utf-8") as f:
        data = json.load(f)
    texts = [d["text"] for d in data]
    metadatas = [d["metadata"] for d in data]
    model = SentenceTransformer(EMBEDDING_MODEL)
    return index, texts, metadatas, model

index, texts, metadatas, model = load_db()

if index is None:
    st.warning("Index non trouvé. Lance les cellules 4 et 6 après avoir placé/téléchargé le PDF dans `data/raw/`.")
    st.info("Source PDF auto-téléchargée depuis: https://www.ipcc.ch/report/ar6/wg1/downloads/report/IPCC_AR6_WGI_Chapter01.pdf")
    st.stop()

question = st.text_input("Pose ta question au rapport du GIEC :", placeholder="Ex: Quelle est la probabilité de dépasser 1.5°C avant 2040 ?")

if question:
    question_embeddings = model.encode([question], normalize_embeddings=True)
    scores, ids = index.search(np.array(question_embeddings, dtype="float32"), k=3)
    docs = [(texts[i], metadatas[i], float(scores[0][rank])) for rank, i in enumerate(ids[0])]

    col1, col2 = st.columns([2, 1])

    with col1:
        st.subheader("Réponse (à brancher à un LLM)")
        st.info(f"{SYSTEM_PROMPT}\n\nContexte récupéré pour: '{question}'")
        for i, (text, meta, score) in enumerate(docs, 1):
            st.markdown(f"**Source {i} - {meta} [score={score:.3f}]**")
            st.write(text)
            st.divider()

    with col2:
        st.subheader("Sources vérifiables")
        for _, meta, _ in docs:
            st.markdown(f"- **{meta['source']}**, p.{meta['page']}")
        st.caption("Branche ensuite un LLM (Mistral/Llama) pour générer la réponse finale avec citations obligatoires.")


---
### Roadmap
- [ ] Placer `IPCC_AR6_WGI_Chapter01.pdf` dans `data/raw/`
- [ ] Executer sections 1 et 2
- [ ] Tester section 3
- [ ] Brancher LLM avec prompt citation obligatoire
